# Web Gold 40K — dataset validation only

This notebook validates the Kaggle dataset `kiyasmahmud/web-gold-40k`. It is intentionally separate from the model-training notebook. It does not call KaggleHub, download or extract the dataset ZIP, train a model, or modify `kaggle_gold.ipynb`.

Before running: use **Add Input** in Kaggle and attach `kiyasmahmud/web-gold-40k`. CPU is sufficient.

In [9]:
# 1. Pull the latest validation code from the Code branch.
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'],
        check=True,
    )
else:
    subprocess.run(
        ['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)],
        check=True,
    )

VALIDATOR = REPO_ROOT / 'scripts' / 'validate_kaggle_gold.py'
assert VALIDATOR.is_file(), f'Validator not found: {VALIDATOR}'
print('Validator:', VALIDATOR)

Updating 1c60623..302e4c6
Fast-forward
 notebooks/kaggle_gold_data_validation.ipynb |  43 ++----------
 project_progress.md                         |  16 +++++
 scripts/validate_kaggle_gold.py             | 100 ++++++++++++++++++++++------
 3 files changed, 104 insertions(+), 55 deletions(-)
Validator: /kaggle/working/webagent/scripts/validate_kaggle_gold.py


From https://github.com/Kiyas-Mahmud/webagent
 * branch            Code       -> FETCH_HEAD
   1c60623..302e4c6  Code       -> origin/Code


In [10]:
# 2. Find the attached dataset folder or ZIP. Nothing is extracted or copied.
INPUT_ROOT = Path('/kaggle/input')
KNOWN_DATASET_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
assert INPUT_ROOT.is_dir(), 'This notebook must run on Kaggle.'

DATA_SOURCE = KNOWN_DATASET_ROOT if KNOWN_DATASET_ROOT.is_dir() else INPUT_ROOT
print('DATA_SOURCE =', DATA_SOURCE)
print('Dataset root entries:')
for item in sorted(DATA_SOURCE.iterdir()):
    kind = 'directory' if item.is_dir() else f'{item.stat().st_size:,} bytes'
    link = f' -> {item.resolve()}' if item.is_symlink() else ''
    print(' -', item.name, f'({kind}){link}')
print('The validator follows version-directory links and streams a nested ZIP in place.')

DATA_SOURCE = /kaggle/input/datasets/kiyasmahmud/web-gold-40k
Dataset root entries:
 - final_data_set_40k (directory)
The validator follows version-directory links and streams a nested ZIP in place.


In [11]:
# 3. Audit settings. The quick run checks every image path and decodes a sample.
# Enable the full hash audit before treating the dataset as publication-ready.
DECODE_SAMPLE = 600
RUN_SHORTCUT_BASELINES = True
FULL_IMAGE_HASH_AUDIT = False  # Set True for exact SHA-256 + near-image dHash checks.
ADULT_DOMAIN_POLICY_CONFIRMED = False  # True only after the thesis protocol records the decision.
REPORT_PATH = Path('/kaggle/working/web_gold_40k_validation_report.json')

print('Decode sample:', DECODE_SAMPLE)
print('Shortcut baselines:', RUN_SHORTCUT_BASELINES)
print('Full image SHA-256 audit:', FULL_IMAGE_HASH_AUDIT)
print('Adult-domain policy confirmed:', ADULT_DOMAIN_POLICY_CONFIRMED)

Decode sample: 600
Shortcut baselines: True
Full image SHA-256 audit: False
Adult-domain policy confirmed: False


In [12]:
# 4. Run the dataset audit. This uses CPU and reads the attached files in place.
command = [
    sys.executable,
    str(VALIDATOR),
    '--data-root', str(DATA_SOURCE),
    '--report', str(REPORT_PATH),
    '--decode-sample', str(DECODE_SAMPLE),
]
if FULL_IMAGE_HASH_AUDIT:
    command.append('--full-image-hash')
if not RUN_SHORTCUT_BASELINES:
    command.append('--skip-baselines')
if ADULT_DOMAIN_POLICY_CONFIRMED:
    command.append('--adult-domain-policy-confirmed')

print('Running:', ' '.join(command))
result = subprocess.run(command, check=False)
assert result.returncode == 0, f'Validator crashed with exit code {result.returncode}'
assert REPORT_PATH.is_file(), f'Report was not created: {REPORT_PATH}'

Running: /usr/bin/python3 /kaggle/working/webagent/scripts/validate_kaggle_gold.py --data-root /kaggle/input/datasets/kiyasmahmud/web-gold-40k --report /kaggle/working/web_gold_40k_validation_report.json --decode-sample 600
DATA_SOURCE = /kaggle/input/datasets/kiyasmahmud/web-gold-40k/final_data_set_40k
No KaggleHub download or ZIP extraction API is used. Files are read in place.
[PASS] nested schema: bad rows=0/39215
[PASS] model input allow-list: rows whose input keys are not exactly ['state_after', 'state_before', 'task_description', 'website_domain']: 0
[PASS] required label fields: all present
[PASS] review_status present: missing=0
[FAIL] approved-only publication gate: pending=39215; approved=0/39215
[PASS] reported split row counts: observed={'train': 23499, 'val': 7861, 'test': 7855}; expected={'train': 23499, 'val': 7861, 'test': 7855}
[PASS] reported total row count: observed=39215; expected=39215
[PASS] outcome label set: FAILURE=22079, SUCCESS=17136
[PASS] failure label se

In [13]:
# 5. Compact report table. Download the JSON report from the Output panel if needed.
import json
import pandas as pd
from IPython.display import display

report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
gate_table = pd.DataFrame(report['gates'])
display(gate_table)

print('Gate counts:', report['gate_counts'])
print('Publication ready:', report['publication_ready'])
print('Saved report:', REPORT_PATH)

blocking = gate_table[gate_table['status'].isin(['FAIL', 'SKIP'])]
if len(blocking):
    print('\nRemaining blocking or skipped gates:')
    display(blocking)

,name,status,detail
0,nested schema,PASS,bad rows=0/39215
1,model input allow-list,PASS,rows whose input keys are not exactly ['state_...
2,required label fields,PASS,all present
3,review_status present,PASS,missing=0
4,approved-only publication gate,FAIL,pending=39215; approved=0/39215
5,reported split row counts,PASS,"observed={'train': 23499, 'val': 7861, 'test':..."
6,reported total row count,PASS,observed=39215; expected=39215
7,outcome label set,PASS,"FAILURE=22079, SUCCESS=17136"
8,failure label set,PASS,"ACTION_MISMATCH=10832, LOOP_DETECTED=1194, NON..."
9,six action classes,PASS,"CLICK=6427, NAVIGATE=7444, PRESS_KEY=5947, SCR..."


Gate counts: {'PASS': 31, 'FAIL': 1, 'WARN': 1, 'SKIP': 3}
Publication ready: False
Saved report: /kaggle/working/web_gold_40k_validation_report.json

Remaining blocking or skipped gates:


,name,status,detail
4,approved-only publication gate,FAIL,pending=39215; approved=0/39215
29,exact image-content split overlap,SKIP,rerun with --full-image-hash for the final pub...
30,near-image perceptual split overlap,SKIP,rerun with --full-image-hash for the dHash Ham...
31,adult-domain thesis protocol,SKIP,"the handoff reports 1,354 rows from 16 adult d..."


## How to interpret the result

- A quick run is expected to show exact and near-image content-overlap gates as **SKIP**. Set `FULL_IMAGE_HASH_AUDIT = True` and rerun cells 4–5 for the final SHA-256 and dHash audit.
- If the uploaded version still has `review_status=pending`, the approved-only publication gate must fail. This is an honest data-governance result, not a notebook error.
- Keep `ADULT_DOMAIN_POLICY_CONFIRMED = False` until the thesis protocol explicitly permits those domains or the 1,354 reported rows are quarantined and replaced.
- The JSON report is small and is written only to `/kaggle/working`; the ZIP remains read-only under `/kaggle/input` and is never extracted.
- Do not start headline training until every required gate passes or each exception is explicitly documented in the thesis protocol.